# Accept / Reject / Uncertain — threshold-based rule categorization

Confidence intervals become a deployment decision tool the moment we pair
them with a threshold `τ`. Each mined rule is then placed in one of three
buckets:

- **Accept** — the entire 95 % CI lies above `τ` (rule is reliably above the
  bar).
- **Reject** — the entire CI lies below `τ` (rule is reliably below the bar).
- **Uncertain** — the CI straddles `τ` (the data don't yet decide).

We run the categorization twice for each dataset: once on the **uplift** CI
(*statistical* significance) and once on the **realistic rule gain** CI
(*economic* significance). The contrast is the headline empirical claim —
a rule can be statistically real yet economically a money-loser.

**Output**: `notebooks/inference_studies/results/categorization_breakdown.csv`.

Bootstrap percentile CIs throughout for consistency with the rest of this
folder.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repository root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

# Import the shared dataset loader (canonical preprocessing for the four
# benchmark datasets used throughout this folder).
sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))
from _datasets import load_all, load_telco, load_bank_marketing, load_employee_attrition, load_credit_card_default  # noqa: E402

# Outputs for this notebook (kept under the notebook folder so the user can
# inspect them locally without touching the article's canonical CSVs).
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'inference_studies' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root:   {REPO_ROOT}')
print(f'Results dir: {RESULTS_DIR.relative_to(REPO_ROOT)}')


In [ ]:
from collections import Counter
from action_rules import ActionRules

OUT_CSV = RESULTS_DIR / 'categorization_breakdown.csv'


In [ ]:
N_BOOTSTRAP = 500
SEED = 42
THRESHOLD = 0.0
METRICS = ['uplift', 'realistic_rule_gain']


In [ ]:
rows = []
for cfg in load_all():
    print(f'=== {cfg.name} ===')
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    ar.fit(
        cfg.df,
        stable_attributes=cfg.stable_attributes,
        flexible_attributes=cfg.flexible_attributes,
        target=cfg.target,
        target_undesired_state=cfg.target_undesired_state,
        target_desired_state=cfg.target_desired_state,
        use_sparse_matrix=cfg.use_sparse_matrix,
    )
    n_rules = len(ar.output.action_rules)
    print(f'  mined {n_rules} rules')

    for metric in METRICS:
        results = ar.confidence_intervals(
            cfg.df,
            method='bootstrap',
            confidence_level=0.95,
            threshold=THRESHOLD,
            metric=metric,
            n_bootstrap=N_BOOTSTRAP,
            random_state=SEED,
        )
        counts = Counter(r.category.value for r in results if r.category is not None)
        rows.append({
            'dataset': cfg.name,
            'metric': metric,
            'n_rules': n_rules,
            'accept': counts.get('accept', 0),
            'uncertain': counts.get('uncertain', 0),
            'reject': counts.get('reject', 0),
            'accept_pct': round(100 * counts.get('accept', 0) / n_rules, 1) if n_rules else 0.0,
            'reject_pct': round(100 * counts.get('reject', 0) / n_rules, 1) if n_rules else 0.0,
        })
        print(f'  {metric:>22}: A={counts.get("accept",0)}, U={counts.get("uncertain",0)}, R={counts.get("reject",0)}')

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)}')
df


## Reading the breakdown

Compare `accept_pct` for `uplift` vs `realistic_rule_gain` within each
dataset. Where the *gain* breakdown shows a non-trivial **reject** count while
the *uplift* breakdown is mostly **accept**, that dataset exhibits the
statistical vs economic significance gap: rules that pass a statistical test
would lose money in deployment because the intervention cost outweighs the
expected outcome change.

IBM Employee Attrition is the cleanest demonstration of this gap (rules
requiring large salary increases tend to have negative realistic gain even
though their uplift is significantly positive).
